In [ ]:
import sys
import json

sys.argv = [
    "main.py",
    "--model", "resnet18_advprop",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "30",
    "--patience", "10",
    "--lr", "0.001",
    "--batch-size", "32",
    "--advprop", 
    "--seed", "42",
    "--epsilon", "0.02",
    "--advprop-iterations","7",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/xie_eps002",
    "--save-plots-path", "outputs/plots/xie_eps002",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18_advprop model...
Model parameters: 11,181,770

Training with AdvProp (epsilon=0.02, epochs=30)...
AdvProp initialized with epsilon=0.02, alpha=0.005

Starting AdvProp Training
Epochs: 30, Epsilon: 0.02
------------------------------------------------------------

Epoch 1/30
--------------------------------------------------


Train Loss: 3.6222
Train Acc (Clean): 36.07%
Train Acc (Adv): 23.54%
Val Loss: 1.2455 | Val Acc: 48.58%
Saved best model with val_acc: 48.58%

Epoch 2/30
--------------------------------------------------


AdvProp Training:  22%|██▏       | 104/473 [01:22<04:51,  1.27it/s, loss=2.86, acc_clean=58.7, acc_adv=27.7]

In [6]:
import os
print("Répertoire actuel:", os.getcwd())
print("\nCherche outputs/models/...")
print("outputs/models existe?", os.path.exists("outputs/models"))
print("../outputs/models existe?", os.path.exists("../outputs/models"))

Répertoire actuel: /home/onyxia/work/Adversarial_examples-1

Cherche outputs/models/...
outputs/models existe? True
../outputs/models existe? False


In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Vérifier le device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Cell 2: Charger le modèle AdvProp entraîné
from models.resnet_with_advprop import ResNet18AdvProp
from train.trainer import create_data_loaders

# Créer le modèle
model = ResNet18AdvProp(num_classes=10)
base_path = "/home/onyxia/work/Adversarial_examples-1"
model_path = os.path.join(base_path, "outputs", "models", "xie_eps002", "best_model_advprop.pth")

# Charger les poids
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"✓ Modèle AdvProp chargé depuis: {model_path}")
print(f"  Epsilon utilisé pendant l'entraînement: {checkpoint.get('epsilon', 'N/A')}")
print(f"  Best val accuracy pendant l'entraînement: {max(checkpoint['history']['val_acc']):.2f}%")

# Cell 3: Charger le dataset test adversarial
test_adv_path = os.path.join(base_path, "datasets", "EuroSAT_RGB", "test_pgd_eps002")
print(f"\nChargement du dataset adversarial: {test_adv_path}")

test_loader, class_names = create_data_loaders(
    data_path=test_adv_path,
    batch_size=32,
    mode="eval"  # Pas de split train/val
)

print(f"✓ Dataset chargé: {len(test_loader.dataset)} images")
print(f"  Classes: {class_names}")

# Cell 4: Fonction d'évaluation
def evaluate_model(model, test_loader, device, use_aux_bn=False):
    """Évalue le modèle sur le dataset donné"""
    model.eval()
    all_preds = []
    all_labels = []
    all_confidences = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc="Evaluation"):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass - IMPORTANT: use_aux_bn=False pour l'évaluation
            outputs = model(inputs, use_aux_bn=use_aux_bn)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            confidences, predicted = torch.max(probabilities, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())
    
    # Calcul des métriques
    accuracy = 100.0 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
    
    return accuracy, all_preds, all_labels, all_confidences

# Cell 5: Évaluation sur le dataset adversarial
print("\n" + "="*60)
print("ÉVALUATION SUR DATASET ADVERSARIAL (test_pgd_eps002)")
print("="*60)

accuracy_adv, preds_adv, labels_adv, confs_adv = evaluate_model(
    model, test_loader, device, use_aux_bn=False
)

print(f"\n✅ Accuracy sur adversarial test: {accuracy_adv:.2f}%")
print(f"   Nombre d'échantillons: {len(labels_adv)}")

Using device: cuda
✓ Modèle AdvProp chargé depuis: /home/onyxia/work/Adversarial_examples-1/outputs/models/xie_eps002/best_model_advprop.pth
  Epsilon utilisé pendant l'entraînement: 0.02
  Best val accuracy pendant l'entraînement: 95.31%

Chargement du dataset adversarial: /home/onyxia/work/Adversarial_examples-1/datasets/EuroSAT_RGB/test_pgd_eps002
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400
✓ Dataset chargé: 5400 images
  Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

ÉVALUATION SUR DATASET ADVERSARIAL (test_pgd_eps002)


Evaluation: 100%|██████████| 169/169 [00:07<00:00, 23.40it/s]


✅ Accuracy sur adversarial test: 38.02%
   Nombre d'échantillons: 5400


In [1]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18_advprop",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/xie_eps002_2",
    "--save-model-path", "outputs/model/xie_eps002",
]

from main import main
main()

Using device: cpu
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/EuroSAT_RGB/train_clean'